# Bias detection using the synthesized data
---
The goal of this notebook is to use different tools to check whether automatic bias detection is possible. We apply it on the original data aswell to have comparable metrics

In [ ]:
import sys

sys.path.append("../src")  # go to parent dir
import warnings
import json
import pandas as pd
from sklearn.linear_model import LogisticRegression
from data_utils import split_data, apply_binning, apply_pdf, convert_intervals
from models_utils import (
    train_and_evaluate_pipeline,
)
from fairness_utils import (
    search_bias,
    evaluate_fairness_score,
    explain_bias,
    encode_protected_attributes,
    search_and_evaluate_fairness,
    train_and_evaluate_fairness_pipeline
)

warnings.filterwarnings("ignore")
random_state = 12041500

## Load data
We start by loading the respective data from the `./data` directory.

In [9]:
def load_data():
    df_train = pd.read_json("../data/synthetic_data_CTGANSynthesizer.json")
    df_test = pd.read_json("../data/testset.json")

    df_train = df_train.drop(columns=["fnlwgt"])
    df_test = df_test.drop(columns=["fnlwgt"])
    return df_train, df_test


ratio_features = ["age", "capital-gain", "capital-loss", "hours-per-week"]
ordinal_features = ["education-num"]
nominal_features = [
    "workclass",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
]
target = "income"

In [10]:
df_train, df_test = load_data()
df_train

,age,workclass,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,income
0,21,Private,9,Never-married,Handlers-cleaners,Husband,Other,Male,0,0,20,0
1,30,Private,6,Married-spouse-absent,Adm-clerical,Not-in-family,White,Female,3,0,40,0
2,47,Private,10,Never-married,Adm-clerical,Not-in-family,Black,Female,0,0,36,0
3,46,Local-gov,14,Divorced,Prof-specialty,Unmarried,White,Female,8,0,40,0
4,63,Private,10,Married-civ-spouse,Farming-fishing,Husband,White,Male,14,0,20,0
...,...,...,...,...,...,...,...,...,...,...,...,...
39068,30,Private,10,Never-married,Other-service,Own-child,Black,Female,4542,0,40,0
39069,32,Private,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,34,0,40,0
39070,27,Private,10,Never-married,Prof-specialty,Other-relative,Other,Male,0,1,40,0
39071,28,Private,6,Never-married,Machine-op-inspct,Other-relative,White,Male,6,0,40,0


## Baseline Classifier
This sections simply trains and evaluates our baseline. We simply use a Logistic Regression because it should be an easy but robust classifier.

In [11]:
clf = LogisticRegression(max_iter=1000, random_state=random_state)

In [13]:
## Train baseline
model, _ = train_and_evaluate_pipeline(
    clf, nominal_features, df_train, df_test, target, drop_na=True, verbose=True
)

Metric          : Value          
Accuracy        : 0.838
Precision       : 0.782
Recall          : 0.746
F1              : 0.761


## Fairness Evaluation
Next, we continue with the evaluation of fairness using multiple approaches.

In [14]:
X_train, y_train = split_data(df_train, target, drop_na=True)
probs = pd.Series(model.predict_proba(X_train)[:, 1])

### Search Bias
We use MDSS to perform a search for privileged classed, concerning the favorable label `>50k`.

In [15]:
privileged_subset, _ = search_bias(X_train, y_train, probs, 1, penalty=1)
print(privileged_subset)

({'capital-gain': [0, 1, 4, 5, 6]}, 226.8525)


In [16]:
_ = evaluate_fairness_score(df_train, privileged_subset[0].keys(), target, verbose=True)

Sensitive Attributes: ['capital-gain']

                         Group Distance  Proportion  Counts   P-Value
capital-gain [37.00, 16383.00]    0.434    0.095449    3563  0.00e+00
    capital-gain [-0.00, 5.00]   -0.137    0.517078   19302 1.48e-323
   capital-gain [23.00, 37.00]    0.253    0.097458    3638 3.41e-256
   capital-gain [15.00, 23.00]    0.087    0.106352    3970  1.82e-38
    capital-gain [5.00, 10.00]   -0.047    0.097645    3645  2.44e-13

Weighted Mean Statistical Distance: 0.1648489023109496


We inspect the subset size, expected probability, and classifier’s probability for protected attributes and privileged classes. We do this to get an better feel for the "problem size" we are facing.

In [17]:
explain_bias(df_train, probs, target, privileged_subset[0])

Our detected privileged group has a size of 19340, we observe 0.0639 as the average probability of earning >50k, but our model predicts 0.2058


## Fairness Metrics
We continue by calculating the fairness metrics: `SPD, EOD, AOD, DI` and `Theil index`. For this we use the AIF360 toolbox. We start by encoding the protected attributes as 1 or 0 based if the respective value is within a privileged class. Furthermore, we have to set those attributes as indices of the dataframe to make it work with the framework.

In [18]:
df_train_bias = encode_protected_attributes(df_train, list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)
df_test_bias = encode_protected_attributes(df_test, list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)

1744 Na rows removed!
202 Na rows removed!


Lastly, we can compute the respective metrics

In [19]:
_, _ = train_and_evaluate_fairness_pipeline(
    clf,
    nominal_features,
    df_train_bias,
    target,
    privileged_subset,
    drop_na=True,
    verbose=True,
)

Metric                         : Value          
statistical_parity_difference   0.221
average_odds_difference         0.184
equal_opportunity_difference    0.302
disparate_impact                5.935
theil_index                     0.113


Next, we continue by looking, how much the fairness metrics depend on the penaltiy parameter

In [20]:
df_fairness_metrics, priviliged_subsets = (
    pd.DataFrame(
        columns=[
            "statistical_parity_difference",
            "average_abs_odds_difference",
            "equal_opportunity_difference",
            "disparate_impact",
            "theil_index",
        ]
    ),
    {},
)

for penalty in [1e-17, 1e-10, 0.001, 0.01, 0.1, 0, 1, 5, 10, 25, 50, 100]:
    metrics, priv = search_and_evaluate_fairness(model, df_train, target, penalty)
    df_fairness_metrics.loc[f"{penalty}"] = metrics.values()
    priviliged_subsets[f"{penalty}"] = priv

In [21]:
df_fairness_metrics.to_json("../results/data/fairness_metrics_synthesized.json")
with open("../results/data/fairness_privileged_classes_synthesized.json", "w") as f:  
    json.dump(priviliged_subsets, f, indent=4)
df_fairness_metrics

,statistical_parity_difference,average_abs_odds_difference,equal_opportunity_difference,disparate_impact,theil_index
1e-17,0.157790,0.301264,0.548028,80.920614,0.113518
1e-10,0.176903,0.143317,0.245121,4.778996,0.113518
0.001,-0.072405,-0.190094,-0.251483,0.678597,0.113518
0.01,0.106165,0.276341,0.548028,3.123298,0.113518
0.1,0.164673,0.148689,0.262768,4.440570,0.113518
0,0.157790,0.301264,0.548028,80.920614,0.113518
1,0.219782,0.182051,0.297375,5.948852,0.113518
5,0.208665,0.180456,0.297944,6.022775,0.113518
10,0.201300,0.189152,0.317138,6.455366,0.113518
25,0.201300,0.189152,0.317138,6.455366,0.113518


## Binned Data
Furthermore, we consider an approach given by AIF360 to use binning. For this, we take their approach 1:1 and check whether it has an influence.

In [22]:
nominal_features = nominal_features + ["age", "hours-per-week", "capital-gain", "capital-loss"]

In [23]:
df_train, df_test = load_data()
df_train = apply_binning(df_train)
df_test = apply_binning(df_test)

Next, we convert all columns with type category to object to avoid problems, which later approaches.

In [24]:
for col in df_train.columns: 
    if col in nominal_features:
        df_train[col] = df_train[col].astype(str)
        df_test[col] = df_test[col].astype(str)

## Baseline Classifier
This sections simply trains and evaluates our baseline. We simply use a Logistic Regression because it should be an easy but robust classifier.

In [25]:
clf = LogisticRegression(max_iter=1000, random_state=random_state)

In [27]:
model, _ = train_and_evaluate_pipeline(
    clf, nominal_features, df_train, df_test, target, drop_na=True, verbose=True
)

Metric          : Value          
Accuracy        : 0.837
Precision       : 0.816
Recall          : 0.701
F1              : 0.732


## Fairness Evaluation
Next, we continue with the evaluation of fairness using multiple approaches.

In [28]:
X_train, y_train = split_data(df_train, target, drop_na=True)
probs = pd.Series(model.predict_proba(X_train)[:, 1])

### Search Bias
We use MDSS to perform a search for privileged classed, concerning the favorable label `>50k`.

In [29]:
privileged_subset, _ = search_bias(X_train, y_train, probs, 1, penalty=1)
print(privileged_subset)

({'education-num': [11, 13, 14], 'capital-loss': ['NoLoss'], 'sex': ['Male'], 'race': ['Black'], 'occupation': ['Prof-specialty', 'Protective-serv']}, 7.357)


In [30]:
_ = evaluate_fairness_score(df_train, privileged_subset[0].keys(), target, verbose=True)

Sensitive Attributes: ['capital-loss', 'education-num', 'occupation', 'race', 'sex']

                               Group Distance  Proportion  Counts   P-Value
        Exec-managerial, White, Male    0.486    0.042460    1585  0.00e+00
         Prof-specialty, White, Male    0.373    0.061052    2279  0.00e+00
                     Exec-managerial    0.409    0.067240    2510  0.00e+00
                Prof-specialty, Male    0.348    0.078464    2929  0.00e+00
               Exec-managerial, Male    0.482    0.051140    1909  0.00e+00
       NoLoss, Exec-managerial, Male    0.472    0.039326    1468  0.00e+00
              Exec-managerial, White    0.417    0.054971    2052  0.00e+00
                              Female   -0.118    0.371481   13867 2.38e-317
             NoLoss, Exec-managerial    0.396    0.051381    1918 2.51e-312
NoLoss, Exec-managerial, White, Male    0.479    0.032495    1213 3.98e-283

Weighted Mean Statistical Distance: 0.13175298419252776


We inspect the subset size, expected probability, and classifier’s probability for protected attributes and privileged classes. We do this to get an better feel for the "problem size" we are facing.

In [31]:
explain_bias(df_train, probs, target, privileged_subset[0])

Our detected privileged group has a size of 110, we observe 0.0455 as the average probability of earning >50k, but our model predicts 0.2658


## Fairness Metrics
We continue by calculating the fairness metrics: `SPD, EOD, AOD, DI` and `Theil index`. For this we use the AIF360 toolbox. We start by encoding the protected attributes as 1 or 0 based if the respective value is within a privileged class. Furthermore, we have to set those attributes as indices of the dataframe to make it work with the framework.

In [32]:
df_train_bias = encode_protected_attributes(df_train, list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)
df_test_bias = encode_protected_attributes(df_test, list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)

Lastly, we can compute the respective metrics

In [33]:
_, _ = train_and_evaluate_fairness_pipeline(
    clf,
    nominal_features,
    df_train_bias,
    target,
    privileged_subset,
    drop_na=True,
    verbose=True,
)

Metric                         : Value          
statistical_parity_difference   0.027
average_odds_difference         -0.229
equal_opportunity_difference    -0.418
disparate_impact                1.200
theil_index                     0.106


Next, we continue by looking, how much the fairness metrics depend on the penaltiy parameter

In [34]:
df_fairness_metrics, priviliged_subsets = (
    pd.DataFrame(
        columns=[
            "statistical_parity_difference",
            "average_abs_odds_difference",
            "equal_opportunity_difference",
            "disparate_impact",
            "theil_index",
        ]
    ),
    {},
)

for penalty in [1e-17, 1e-10, 0.001, 0.01, 0.1, 0, 1, 5, 10, 25, 50, 100]:
    metrics, priv = search_and_evaluate_fairness(model, df_train, target, penalty)
    df_fairness_metrics.loc[f"{penalty}"] = metrics.values()
    priviliged_subsets[f"{penalty}"] = priv

In [35]:
df_fairness_metrics.to_json("../results/data/fairness_metrics_synthesized_binning.json")
with open("../results/data/fairness_privileged_classes_synthesized_binning.json", "w") as f:  
    json.dump(priviliged_subsets, f, indent=4)

df_fairness_metrics

,statistical_parity_difference,average_abs_odds_difference,equal_opportunity_difference,disparate_impact,theil_index
1e-17,0.120956,0.297646,0.582492,3.797097,0.106032
1e-10,0.152072,0.313252,0.582492,13.469931,0.106032
0.001,0.152072,0.313252,0.582492,13.469931,0.106032
0.01,0.151993,0.313214,0.582492,13.387437,0.106032
0.1,0.161054,0.317384,0.582492,36.995486,0.106032
0,0.167404,0.320067,0.583104,178.281059,0.106032
1,0.027316,-0.228598,-0.417781,1.200319,0.106032
5,0.000000,0.000000,0.000000,1.000000,0.000000
10,0.000000,0.000000,0.000000,1.000000,0.000000
25,0.000000,0.000000,0.000000,1.000000,0.000000


## PDFed Data
Additionally, we propose an alternative approach using PDFs to encode the same features. This method groups data based on the actual distribution rather than discrete bins, resulting in a smooth curve that represents the distribution continuously. This may offer
a potentially more effective solution.

In [36]:
nominal_features = nominal_features + ["age", "hours-per-week"]

In [37]:
df_train, df_test = load_data()
df_train = apply_pdf(df_train.dropna(), target)
df_test = apply_pdf(df_test.dropna(), target)

Next, we convert all columns with type category to object to avoid problems, which later approaches.

In [38]:
for col in df_train.columns:
    if col in nominal_features:
        df_train[col] = df_train[col].astype("object")
        df_test[col] = df_test[col].astype("object")

### Baseline Classifier
This sections simply trains and evaluates our baseline. We simply use a Logistic Regression because it should be an easy but robust classifier.

In [39]:
clf = LogisticRegression(max_iter=1000, random_state=random_state)

In [40]:
## Train baseline
model, _ = train_and_evaluate_pipeline(
    clf, nominal_features, df_train, df_test, target, drop_na=True, verbose=True
)

Metric          : Value          
Accuracy        : 0.755
Precision       : 0.714
Recall          : 0.782
F1              : 0.720


### Fairness Evaluation
Next, we continue with the evaluation of fairness using multiple approaches.

In [41]:
X_train, y_train = split_data(df_train, target, drop_na=True)
probs = pd.Series(model.predict_proba(X_train)[:, 1])

### Search Bias
We use MDSS to perform a search for privileged classed, concerning the favorable label `>50k`.

In [42]:
privileged_subset, _ = search_bias(X_train, y_train, probs, 1, penalty=1)
print(privileged_subset)

({'capital-gain': [0.0014268116068545864, 0.0014494447135108503, 0.0014499044018283546, 0.0014502686425889616, 0.0014506540370765706, 0.0014508448770105507], 'workclass': ['?', 'Local-gov'], 'age': [0.0047791226256496434, 0.006711487346189525, 0.006959676452913946, 0.0074541789835679915, 0.009036480031066975, 0.03695082496929242, 0.03792455714871646]}, 11.1823)


In [43]:
_ = evaluate_fairness_score(df_train, privileged_subset[0].keys(), target, verbose=True)

Sensitive Attributes: ['age', 'capital-gain', 'workclass']

                                       Group Distance  Proportion  Counts   P-Value
                  capital-gain [-0.00, 0.00]    0.435    0.100029    3734  0.00e+00
         capital-gain [-0.00, 0.00], Private    0.431    0.061963    2313  0.00e+00
age [0.01, 0.02], capital-gain [-0.00, 0.00]    0.548    0.014921     557 2.95e-169
age [0.02, 0.03], capital-gain [-0.00, 0.00]    0.548    0.014278     533 4.20e-162
                                Self-emp-inc    0.338    0.032629    1218 1.36e-148
    capital-gain [-0.00, 0.00], Self-emp-inc    0.605    0.010099     377 1.41e-139
                capital-gain [0.00, 0.00], ?   -0.161    0.069865    2608 3.48e-126
age [0.01, 0.01], capital-gain [-0.00, 0.00]    0.497    0.013100     489 1.35e-123
                                           ?   -0.155    0.073026    2726 1.26e-119
                   capital-gain [0.00, 0.00]   -0.048    0.899971   33595 8.34e-114

Weighted Mean S

We inspect the subset size, expected probability, and classifier’s probability for protected attributes and privileged classes. We do this to get an better feel for the "problem size" we are facing.

In [44]:
explain_bias(df_train, probs, target, privileged_subset[0])

Our detected privileged group has a size of 374, we observe 0.0 as the average probability of earning >50k, but our model predicts 0.2062


### Fairness Metrics
We continue by calculating the fairness metrics: `SPD, EOD, AOD, DI` and `Theil index`. For this we use the AIF360 toolbox. We start by encoding the protected attributes as 1 or 0 based if the respective value is within a privileged class. Furthermore, we have to set those attributes as indices of the dataframe to make it work with the framework.

In [45]:
df_train_bias = encode_protected_attributes(df_train, list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)
df_test_bias = encode_protected_attributes(df_test, list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)

Lastly, we can compute the respective metrics

In [46]:
_, _ = train_and_evaluate_fairness_pipeline(
    clf,
    nominal_features,
    df_train_bias,
    target,
    privileged_subset,
    drop_na=True,
    verbose=True,
)

Metric                         : Value          
statistical_parity_difference   0.151
average_odds_difference         0.325
equal_opportunity_difference    0.616
disparate_impact                10.439
theil_index                     0.097


Next, we continue by looking, how much the fairness metrics depend on the penaltiy parameter

In [47]:
df_fairness_metrics, priviliged_subsets = (
    pd.DataFrame(
        columns=[
            "statistical_parity_difference",
            "average_abs_odds_difference",
            "equal_opportunity_difference",
            "disparate_impact",
            "theil_index",
        ]
    ),
    {},
)

for penalty in [1e-17, 1e-10, 0.001, 0.01, 0.1, 0, 1, 5, 10, 25, 50, 100]:
    metrics, priv = search_and_evaluate_fairness(model, df_train, target, penalty)
    df_fairness_metrics.loc[f"{penalty}"] = metrics.values()
    priviliged_subsets[f"{penalty}"] = priv

In [48]:
df_fairness_metrics.to_json("../results/data/fairness_metrics_synthesized_pdf.json")
with open("../results/data/fairness_privileged_classes_synthesized_pdf.json", "w") as f:  
    json.dump(priviliged_subsets, f, indent=4)

df_fairness_metrics

,statistical_parity_difference,average_abs_odds_difference,equal_opportunity_difference,disparate_impact,theil_index
1e-17,0.108992,0.304078,0.615647,2.884921,0.097047
1e-10,0.128299,0.313844,0.615647,4.335786,0.097047
0.001,0.104304,0.301918,0.615647,2.678353,0.097047
0.01,0.123871,0.311769,0.615647,3.910979,0.097047
0.1,0.095415,0.297550,0.615647,2.346415,0.097047
0,0.076694,-0.210057,-0.384404,1.854590,0.097047
1,0.151431,0.325225,0.615647,10.439210,0.097047
5,0.000000,0.000000,0.000000,1.000000,0.000000
10,0.000000,0.000000,0.000000,1.000000,0.000000
25,0.000000,0.000000,0.000000,1.000000,0.000000
